In [1]:
import numpy as np
from matplotlib import pyplot as plt

In [2]:
from scipy.integrate import solve_ivp

In [6]:
M = 10
a = 1

In [7]:
def unpack(y):
    r1 = np.array([y[0],y[1],y[2]])
    v1 = np.array([y[3],y[4],y[5]])
    return r1,v1

In [20]:
r?

Signature: r(r, a=1)
Docstring: <no docstring>
File:      /tmp/ipykernel_150288/3067149130.py
Type:      function

In [21]:
def K(r,a=a):
    x,y,z = r[0],r[1],r[2]
    return (x**2+y**2+z**2-a**2)/2
    
def W(r,a=a):
    x,y,z = r[0],r[1],r[2]
    return np.sqrt(K(r)**2+a**2*z**2)

def rkw(r, a=a):
    return np.sqrt(K(r)+W(r))

In [27]:
def dx_K(r,a=a):
    x,y,z = r[0],r[1],r[2]
    return x
def dy_K(r,a=a):
    x,y,z = r[0],r[1],r[2]
    return y
def dz_K(r,a=a):
    x,y,z = r[0],r[1],r[2]
    return z

def di_K(r,a=a):
    return np.array([dx_K(r), dy_K(r), dz_K(r)])
    


def dx_W(r,a=a):
    x,y,z = r[0],r[1],r[2]
    return dx_K(r)/W(r)

def dy_W(r,a=a):
    x,y,z = r[0],r[1],r[2]
    return dy_K(r)/W(r)

def dz_W(r,a=a):
    x,y,z = r[0],r[1],r[2]
    return z*(K(r)+a**2)/W(r)

def di_W(r,a=a):
    return np.array([dx_W(r), dy_W(r), dz_W(r)])

In [28]:
def rho2(r,a=a):
    x,y,z = r[0],r[1],r[2]
    return rkw(r)+a**2

def dx_r(r,a=a):
    x,y,z = r[0],r[1],r[2]
    return x*rkw(r)/(2*W(r))

def dy_r(r,a=a):
    x,y,z = r[0],r[1],r[2]
    return y*rkw(r)/(2*W(r))

def dz_r(r,a=a):
    x,y,z = r[0],r[1],r[2]
    return z*rho2(r)/(2*r*W(r))

def di_r(r,a=a):
    return np.array([dx_r(r), dy_r(r), dz_r(r)])

In [29]:
def dx_S(r,a=a):
    x,y,z = r[0],r[1],r[2]
    return 2*x*rkw(r)**4/(W(r))

def dy_S(r,a=a):
    x,y,z = r[0],r[1],r[2]
    return 2*y*rkw(r)**4/(W(r))

def dx_S(r,a=a):
    x,y,z = r[0],r[1],r[2]
    return 2*x*rkw(r)**2*rho2(r)**2/(W(r))+2*a**2*z


def di_S(r,a=a):
    return np.array([dx_S(r), dy_S(r), dz_S(r)])

In [30]:
def f(r,a=a):
    x,y,z = r[0],r[1],r[2]
    return M*rkw(r)/W(r)
    
def dx_f(r,a=a):
    x,y,z = r[0],r[1],r[2]
    return M/W(r) * (dx_r(r)-rkw(r)/W(r)*dx_W(r))

def dy_f(r,a=a):
    x,y,z = r[0],r[1],r[2]
    return M/W(r) * (dy_r(r)-rkw(r)/W(r)*dy_W(r))

def dz_f(r,a=a):
    x,y,z = r[0],r[1],r[2]
    return M/W(r) * (dz_r(r)-rkw(r)/W(r)*dz_W(r))


def di_f(r,a=a):
    return np.array([dx_f(r), dy_f(r), dz_f(r)])

In [35]:
def Lx(r, a=a):
    x,y,z = r[0],r[1],r[2]
    return (x*(a**2-rkw(r)**2) - 2*a*r*y)/(rho(r)**4)

def Ly(r, a=a):
    x,y,z = r[0],r[1],r[2]
    return (y*(a**2-rkw(r)**2) - 2*a*r*x)/(rho(r)**4)

def Lz(r, a=a):
    x,y,z = r[0],r[1],r[2]
    return -z/rkw(r)**2

def L(r,a=a):
    return np.array([Lx(r), Ly(r), Lz(r)])


def l(r,a=a):
    '''indici bassi'''
    lt = 1
    lx = (rkw(r)*x+a*y)/rho2
    ly = (rkw(r)*y-a*x)/rho2
    lz = z/rkw(r)
    return np.array([lt, lx, ly, lz])

def di_lx(r,a=a):
    dx_lx = rkw(r)/rho(r)**2 + Lx(r)*dx_r(r)
    dy_lx = a/rho(r)**2 + Lx(r)*dy_r(r)
    dz_lx = Lx(r)*dz_r(r)
    return np.array([dx_lx, dy_lx, dz_lx])

def di_lx(r,a=a):
    dy_ly = rkw(r)/rho(r)**2 + Ly(r)*dy_r(r)
    dx_ly = -a/rho(r)**2 + Ly(r)*dx_r(r)
    dz_ly = Ly(r)*dz_r(r)
    return np.array([dx_ly, dy_ly, dz_ly])

def di_lz(r,a=a):
    dx_lz = Lx(r)*dx_r(r)
    dy_lz = Lz(r)*dy_r(r)
    dz_lz = 1/rkw(r)+Lz(r)*dz_r(r)
    return np.array([dx_lz, dy_lz, dz_lz])

def deriv_l(r,a=a):
    di_lt = np.array([0,0,0])
    Dl =  np.array([di_lt, di_lx(r), di_ly(r), di_lz(r)])
    Dl = Dl.T
    return Dl

In [ ]:
#np.einsum

In [46]:
def deriv_gmn344(r,a=a):
    D_g = np.zeros((3,4,4))
    for k in range(3):
        for mu in range(4):
            for nu in range(4):
                D_g[k, mu, nu] = di_f(r)[k] * l(r)[mu] * l(r)[nu]

    return D_g

def deriv_gmn444(r,a=a):
    D_g = np.zeros((4,4,4))
    for k in range(1,4):
        for mu in range(4):
            for nu in range(4):
                D_g[k, mu, nu] = di_f(r)[k] * l(r)[mu] * l(r)[nu]
                D_g[0, mu, nu] = 0

    return D_g

In [49]:
def G(r,a=a):
    Galphabeta = np.zeros((4,4))
    Gabs = np.zeros((4,4,4))
    for alpha in range(4):
        for beta in range(4):
            for sigma in range(4):
                Gabs[alpha,beta,sigma] = l(r)[sigma] * ( deriv_gmn444(r)[alpha,beta,sigma] + deriv_gmn444(r)[beta,alpha,sigma] - deriv_gmn444(r)[sigma,alpha,beta])
                
    Galphabeta[alpha, beta] = np.einsum('ijk->ij', Gabs)
    return Galphabeta